# 04 — Error types of the harmful flips (LLM judge + hand check)

Gives every **harmful flip** from notebook 03 (FP16 right → quantized wrong) one **error type**, then tests
whether RTN and GPTQ make **different kinds** of mistakes (**RQ2**).

**How it works**
1. Flips already labelled by rules in notebook 03 (degeneration, no answer) keep their label.
2. The rest go to **one LLM judge** with fixed definitions. The judge is *blind*: it never sees which model wrote the answer.
3. You **hand-label a random sample** (a CSV this notebook creates). Agreement with the judge = **Cohen's κ**.
4. **χ² test:** is the mix of error types different for RTN and GPTQ?

**Before running**
- **Input:** attach the **03_Flips** notebook (or a dataset with `03_harmful_flips.jsonl`): **+ Add Input**.
- **API keys:** **Add-ons → Secrets → Add Secret** for each provider in `JUDGES` (cell 1). Tick them for this notebook.
- **One judge per task** (`JUDGES`), all in one run. If one provider hits its daily limit, the other task still finishes;
  run again later (with this notebook attached as an Input) and only the missing flips are sent.
- A new `TAG` gives a new set of files, so earlier results are never touched.
- **Internet: On** (Settings). Accelerator: **None**.
- Set `JUDGES` in cell 1. Put `""` as a model once to print the models that key can use.

**Order of runs**
1. `RUN = "pilot"`: judges only the hand-label sample (~70 flips). Download `04_hand_sample.csv`, fill the `my_label` column,
   upload it as a dataset named e.g. `hand-labels` (file name must contain `hand_labels`), attach it → κ is computed.
2. `RUN = "all"`: judges every flip. **Checkpointed**: if it stops (daily limit, crash), attach this notebook's last
   version as an Input and run again; it continues where it stopped.

**Saved files:** `04_judged.jsonl` (judge answers), `04_labels.jsonl` (final label of every flip),
`04_hand_sample.csv` (for you to fill), `04_results.json` (κ, label counts, χ²).

In [ ]:
# 1. Settings
# One judge per task: (provider, model, Kaggle Secret with that provider's key)
#   provider: "gemini", "groq", "openrouter", or "fake" (random labels, only to test the notebook)
JUDGES = {
    "triviaqa": ("gemini", "gemini-3.5-flash-lite", "JUDGE_API_KEY"),
    "gsm8k":    ("groq",   "openai/gpt-oss-120b",   "GROQ_API_KEY"),
}
TASKS      = ["triviaqa", "gsm8k"]   # tasks this run judges and analyses (in this order)
TAG        = "3b"             # names this run's files (04_judged_3b.jsonl, ...); "" = the original 04 files
MODELS_ONLY = None            # judge only these models' flips (None = every model in the flips file)
RUN        = "all"        # "pilot" = only the hand-label sample, "all" = every flip
REASONING  = "medium"     # gpt-oss only: "low", "medium" or "high" (more = better but more tokens)
BATCH      = {"gsm8k": 4, "triviaqa": 10}           # flips per request
MAX_REQUESTS = {"gemini": 450, "groq": 900, "openrouter": 900, "fake": 10**9}   # per provider, per run
SLEEP_S    = {"gemini": 4.5, "groq": 40, "openrouter": 4.5, "fake": 0}         # pause between requests
HAND_N     = {"gsm8k": 30, "triviaqa": 40}  # hand-label sample size per task
SEED       = 0
OUT_DIR    = "/kaggle/working"

BASE_URLS = {
    "gemini":     "https://generativelanguage.googleapis.com/v1beta/openai",
    "groq":       "https://api.groq.com/openai/v1",
    "openrouter": "https://openrouter.ai/api/v1",
}

In [ ]:
# 2. Load the harmful flips (+ any earlier judge answers, so a stopped run continues)
import glob, json, os, re, shutil, random, time
import numpy as np, pandas as pd

def read_jsonl(path):
    rows = []
    if path and os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        rows.append(json.loads(line))
                    except json.JSONDecodeError:
                        pass
    return rows

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return hits[0] if hits else None

src = find("03_harmful_flips*")
assert src, "Attach notebook 03's output (03_harmful_flips.jsonl) as an Input"
flips = read_jsonl(src)
if MODELS_ONLY:
    flips = [r for r in flips if r["model"] in MODELS_ONLY]
    assert flips, f"no flips of {MODELS_ONLY} in {src}: attach the notebook-03 version that includes them"
for r in flips:
    r["key"] = f"{r['task']}|{r['model']}|{r['id']}"
print(len(flips), "harmful flips from", src)

SUF = f"_{TAG}" if TAG else ""
JUDGED = f"{OUT_DIR}/04_judged{SUF}.jsonl"
if not os.path.exists(JUDGED) and find(f"04_judged{SUF}.jsonl"):
    shutil.copy(find(f"04_judged{SUF}.jsonl"), JUDGED); print(f"restored earlier judge answers (04_judged{SUF}.jsonl)")
judged = {r["key"]: r for r in read_jsonl(JUDGED)}
print(len(judged), "flips already judged")

df = pd.DataFrame(flips)
print(pd.crosstab([df.task, df.model], df.rule_label))

In [ ]:
# 3. Hand-label sample: a random, blind sample of judge-bound flips for YOU to label
#    (model names hidden; the answer key is kept in a separate file)
SAMPLE = f"{OUT_DIR}/04_hand_sample.csv"
KEYMAP = f"{OUT_DIR}/04_hand_sample_key.csv"
if not os.path.exists(SAMPLE) and find("04_hand_sample.csv") and find("04_hand_sample_key.csv"):
    shutil.copy(find("04_hand_sample.csv"), SAMPLE); shutil.copy(find("04_hand_sample_key.csv"), KEYMAP)

if not os.path.exists(SAMPLE):
    rng = random.Random(SEED)
    rows, keys = [], []
    for task, n in HAND_N.items():
        pool = [r for r in flips if r["task"] == task and r["rule_label"] == "judge"]
        for r in rng.sample(pool, min(n, len(pool))):
            sid = f"S{len(rows) + 1:03d}"
            rows.append({"sample_id": sid, "task": task, "question": r["question"], "gold": r["gold"],
                         "correct_answer_fp16": r["fp16_output"], "wrong_answer": r["quant_output"], "my_label": ""})
            keys.append({"sample_id": sid, "key": r["key"]})
    rng.shuffle(rows)
    pd.DataFrame(rows).to_csv(SAMPLE, index=False)
    pd.DataFrame(keys).to_csv(KEYMAP, index=False)
    print("created", SAMPLE)
hand_keys = set(pd.read_csv(KEYMAP)["key"])
print(len(hand_keys), "flips in the hand-label sample")
print("Labels to use -> math: conceptual, method, execution, reasoning, degeneration, scoring_issue")
print("                 facts: near_miss, unrelated, no_answer, degeneration, scoring_issue")

In [ ]:
# 4. Judge prompt (fixed definitions; the judge never sees which model wrote the answer)
LABELS = {
    "gsm8k": {
        "conceptual":    "misreads, ignores or invents information in the question (forgets a quantity, uses the wrong number from the text, misreads a condition or what is asked), even if the arithmetic after that is correct",
        "method":        "reads the question correctly but uses a wrong plan, formula, or order of operations",
        "execution":     "the set-up uses the right numbers and relationships, but an arithmetic or copying mistake in a calculation gives the wrong number",
        "reasoning":     "a step does not follow from the earlier ones, contradicts itself, or the final answer does not match its own work",
        "degeneration":  "repeats itself, gets cut off, or never gives a final answer",
        "scoring_issue": "the answer is actually correct (or the gold answer is wrong/ambiguous); the automatic check was wrong",
    },
    "triviaqa": {
        "near_miss":     "a related but wrong answer of the same kind (another city in the same country, a related person, a nearby year)",
        "unrelated":     "a wrong answer with no clear link to the correct one, or of a different kind",
        "no_answer":     "refuses, says it does not know, or repeats the question",
        "degeneration":  "junk text, loops, or a long ramble without a clear answer",
        "scoring_issue": "the answer is actually correct (synonym, alias, more specific form); the automatic check missed it",
    },
}
TASK_NAME = {"gsm8k": "grade-school math word problems", "triviaqa": "trivia questions answered from memory"}
HINTS = {
    "gsm8k": ("How to tell conceptual from execution: compare the wrong answer's set-up with the correct reference answer. "
              "If it left out, added or misread a number or condition from the question, it is conceptual (even if every "
              "calculation after that is right). Only if the set-up matches and a calculation result is wrong is it execution.\n\n"),
    "triviaqa": "",
}

def clip(s, n):
    s = " ".join(str(s).split())
    return s if len(s) <= n else s[: n // 2] + "  [... middle cut ...]  " + s[-n // 2:]

def make_prompt(task, batch):
    n = 2400 if task == "gsm8k" else 400
    defs = "\n".join(f"- {k}: {v}" for k, v in LABELS[task].items())
    items = "\n\n".join(
        f"### Item {i + 1}\nQuestion: {clip(r['question'], 1200)}\nGold answer: {r['gold']}\n"
        f"Correct reference answer: {clip(r['fp16_output'], n)}\nAnswer to classify (judged WRONG): {clip(r['quant_output'], n)}"
        for i, r in enumerate(batch))
    return (f"You are an expert judge for a research study on the errors language models make. "
            f"Your job is to classify why an answer to {TASK_NAME[task]} is wrong. Be careful, strict and consistent: "
            f"the same kind of mistake must always get the same label.\n\n"
            f"Each item shows the question, the gold answer, a correct reference answer, and the wrong answer to classify. "
            f"Give each item exactly ONE label:\n{defs}\n\n{HINTS[task]}"
            f"If several fit, choose the FIRST mistake that made the answer wrong. "
            f"Reply with JSON only, no other text: "
            f'{{"labels": [{{"item": 1, "label": "<label>", "reason": "<at most 15 words>"}}, ...]}} '
            f"with one entry per item.\n\n{items}")

print(make_prompt(TASKS[0], [r for r in flips if r["task"] == TASKS[0]][:1]))

In [ ]:
# 5. API client (OpenAI-compatible chat endpoint) + model list
import requests

HEADS = {}
for task in TASKS:
    provider, model, secret = JUDGES[task]
    if provider == "fake" or provider in HEADS:
        continue
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret(secret)
    HEADS[provider] = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    if not model:
        r = requests.get(f"{BASE_URLS[provider]}/models", headers=HEADS[provider], timeout=60)
        print(provider, r.status_code)
        names = [m.get("id") for m in r.json().get("data", [])] if r.ok else []
        print("\n".join(sorted(names)) if names else r.text[:2000])
        raise SystemExit(f"Pick a model for {task} from the list above (cell 1), then run again.")
print("judges:", {t: f"{JUDGES[t][0]}:{JUDGES[t][1]}" for t in TASKS})

class DailyLimit(Exception):
    pass

def call_judge(prompt, task):
    provider, model, _ = JUDGES[task]
    if provider == "fake":
        return None
    BASE, HEAD = BASE_URLS[provider], HEADS[provider]
    body = {"model": model, "temperature": 0, "messages": [{"role": "user", "content": prompt}]}
    if "gpt-oss" in model:
        body["reasoning_effort"] = REASONING
    for attempt in range(6):
        try:
            r = requests.post(f"{BASE}/chat/completions", headers=HEAD, json=body, timeout=180)
        except requests.RequestException as e:
            print("  network error:", e); time.sleep(10 * (attempt + 1)); continue
        if r.ok:
            return r.json()["choices"][0]["message"]["content"]
        txt = r.text[:300]
        if r.status_code == 429 and re.search(r"per.?day|daily|RPD|quota", txt, re.I):
            raise DailyLimit(txt)
        if r.status_code in (429, 500, 502, 503, 504):
            wait = 15 * (attempt + 1); print(f"  {r.status_code}, waiting {wait}s"); time.sleep(wait); continue
        raise RuntimeError(f"{r.status_code}: {txt}")
    raise RuntimeError("too many failed attempts")

def parse(reply, task, n):
    """Return {item_number: (label, reason)} for the valid entries in the judge's reply."""
    if reply is None:                                   # fake judge
        return {i + 1: (random.choice(list(LABELS[task])), "fake") for i in range(n)}
    m = re.search(r"\{.*\}", reply, re.S)
    if not m:
        return {}
    try:
        data = json.loads(m.group(0))
    except json.JSONDecodeError:
        return {}
    out = {}
    for e in data.get("labels", []):
        lab = str(e.get("label", "")).strip().lower().replace("-", "_").replace(" ", "_")
        try:
            i = int(e.get("item"))
        except (TypeError, ValueError):
            continue
        if lab in LABELS[task] and 1 <= i <= n:
            out[i] = (lab, str(e.get("reason", ""))[:200])
    return out

In [ ]:
# 6. Run the judge (checkpointed: every answer is appended to 04_judged.jsonl right away)
todo = [r for r in flips if r["task"] in TASKS and r["rule_label"] == "judge" and r["key"] not in judged
        and (RUN == "all" or r["key"] in hand_keys)]
random.Random(SEED).shuffle(todo)                       # mix RTN and GPTQ flips in every request
print(f"{len(todo)} flips to judge in this run ({RUN})")

made, t0 = {}, time.time()
with open(JUDGED, "a") as f:
    for task in TASKS:
        provider, model, _ = JUDGES[task]
        items = [r for r in todo if r["task"] == task]
        print(f"{task}: {len(items)} flips -> {provider}:{model}")
        try:
            for i in range(0, len(items), BATCH[task]):
                if made.get(provider, 0) >= MAX_REQUESTS[provider]:
                    raise DailyLimit(f"reached MAX_REQUESTS = {MAX_REQUESTS[provider]} for {provider}")
                batch = items[i:i + BATCH[task]]
                reply = call_judge(make_prompt(task, batch), task); made[provider] = made.get(provider, 0) + 1
                got = parse(reply, task, len(batch))
                for k, r in enumerate(batch):
                    if k + 1 in got:
                        lab, why = got[k + 1]
                        row = {"key": r["key"], "task": task, "model": r["model"], "judge_label": lab, "reason": why,
                               "judge": f"{provider}:{model}"}
                        judged[r["key"]] = row; f.write(json.dumps(row) + "\n")
                f.flush()
                if made[provider] % 20 == 0 or i + BATCH[task] >= len(items):
                    print(f"  {task}: {min(i + BATCH[task], len(items))}/{len(items)} | {made[provider]} {provider} requests | {(time.time()-t0)/60:.1f} min")
                time.sleep(SLEEP_S[provider])
        except DailyLimit as e:
            print(f"STOPPED {task}:", str(e)[:200])
            print("-> the other tasks continue; next time attach this notebook as an Input and run again.")

need = [r for r in flips if r["task"] in TASKS and r["rule_label"] == "judge" and (RUN == "all" or r["key"] in hand_keys)]
left = sum(r["key"] not in judged for r in need)
print(f"\njudged so far: {len(judged)} | still missing in this run's scope: {left}")

In [ ]:
# 7. Final label of every flip (rule label if notebook 03 gave one, otherwise the judge's)
final = []
for r in [r for r in flips if r["task"] in TASKS]:
    if r["rule_label"] != "judge":
        lab, source = r["rule_label"], "rule"
    elif r["key"] in judged:
        lab, source = judged[r["key"]]["judge_label"], "judge"
    else:
        lab, source = None, "missing"
    final.append({"key": r["key"], "task": r["task"], "model": r["model"], "id": r["id"], "label": lab, "source": source,
                  "reason": judged.get(r["key"], {}).get("reason", "")})
with open(f"{OUT_DIR}/04_labels{SUF}.jsonl", "w") as f:
    for r in final:
        f.write(json.dumps(r) + "\n")
lab_df = pd.DataFrame(final)
print(lab_df.groupby(["task", "source"]).size())

In [ ]:
# 8. Agreement between the judge and your hand labels (Cohen's kappa)
from sklearn.metrics import cohen_kappa_score, confusion_matrix
results = {"judge": {t: f"{JUDGES[t][0]}:{JUDGES[t][1]}" for t in TASKS}, "tasks": TASKS, "kappa": {}}

hl = find("*hand_labels*.csv")
if not hl:
    print("No hand labels attached yet. Download 04_hand_sample.csv from the Output, fill 'my_label',")
    print("save it as 04_hand_labels.csv, upload it as a dataset and attach it here.")
else:
    mine = pd.read_csv(hl).merge(pd.read_csv(KEYMAP), on="sample_id")
    mine["my_label"] = mine["my_label"].astype(str).str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")
    mine["judge_label"] = mine["key"].map(lambda k: judged.get(k, {}).get("judge_label"))
    mine = mine[mine["task"].isin(TASKS) & mine["my_label"].isin(set().union(*[set(v) for v in LABELS.values()]))
                & mine["judge_label"].notna()]
    for task, g in mine.groupby("task"):
        k = cohen_kappa_score(g["my_label"], g["judge_label"])
        agree = (g["my_label"] == g["judge_label"]).mean()
        results["kappa"][task] = {"kappa": k, "raw_agreement": agree, "n": len(g)}
        print(f"\n{task}: kappa = {k:.2f} | raw agreement = {agree:.0%} | n = {len(g)}")
        labs = list(LABELS[task])
        print(pd.DataFrame(confusion_matrix(g["my_label"], g["judge_label"], labels=labs),
                           index=[f"me:{l}" for l in labs], columns=[f"judge:{l}" for l in labs]))
    print("\nRule of thumb: kappa > 0.6 substantial, 0.4-0.6 moderate, < 0.4 weak")

In [ ]:
# 9. RQ2 - do RTN and GPTQ make different kinds of mistakes? (chi-square test)
from scipy.stats import chi2_contingency
results["rq2"] = {}
done = lab_df[lab_df["label"].notna()]
for task in TASKS:
    g = done[done.task == task]
    if g.empty:
        continue
    tab = pd.crosstab(g["model"], g["label"])
    pct = (100 * tab.div(tab.sum(axis=1), axis=0)).round(1)
    print(f"\n=== {task}: counts ===\n{tab}\n\n{task}: % of each model's harmful flips\n{pct}")
    real = tab.drop(columns=[c for c in ["scoring_issue"] if c in tab.columns])
    real = real.loc[:, real.sum() > 0]
    entry = {"counts": tab.to_dict(), "percent": pct.to_dict(),
             "missing": int((lab_df.task.eq(task) & lab_df.label.isna()).sum())}
    if real.shape[0] == 2 and real.shape[1] >= 2:
        chi2, p, dof, exp = chi2_contingency(real)
        entry.update({"chi2": chi2, "p": p, "dof": dof, "min_expected": float(exp.min())})
        print(f"\nchi-square (without scoring_issue): chi2 = {chi2:.2f}, dof = {dof}, p = {p:.4f}"
              + ("   (some expected counts < 5: read with care)" if exp.min() < 5 else ""))
    results["rq2"][task] = entry
    if entry["missing"]:
        print(f"note: {entry['missing']} {task} flips not judged yet")

In [ ]:
# 10. Save
def clean(o):
    if isinstance(o, dict):  return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o
with open(f"{OUT_DIR}/04_results{SUF}.json", "w") as f:
    json.dump(clean(results), f, indent=1)
print(f"saved: 04_judged{SUF}.jsonl, 04_labels{SUF}.jsonl, 04_results{SUF}.json")